In [43]:
!pip install transformers datasets huggingface_hub transformers[torch] accelerate --upgrade

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

In [45]:
from huggingface_hub import login

login()

In [46]:
import re
from sklearn.model_selection import train_test_split

In [47]:
f = open("./drive/MyDrive/metriccoders_datasets/history_of_mysore.txt", "r")
text = f.readlines()

In [48]:
print(len(text))

623


In [49]:
def build_text_files(data_text, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for texts in data_text:
        summary = str(texts).strip()
        summary = re.sub(r"\s", " ", summary)
        data += summary + "  "
    f.write(data)

train, test = train_test_split(text,test_size=0.15)


build_text_files(train,'train_dataset.txt')
build_text_files(test,'test_dataset.txt')

print("Train dataset length: "+str(len(train)))
print("Test dataset length: "+ str(len(test)))

Train dataset length: 529
Test dataset length: 94


In [50]:
tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-small")

In [51]:
train_path = "train_dataset.txt"
test_path = "test_dataset.txt"

In [52]:
from transformers import TextDataset, DataCollatorForLanguageModeling
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [53]:
def load_dataset(train_path, test_path, tokeinzer):
  train_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=train_path,
          block_size=64)
  test_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=test_path,
          block_size=64)
  data_collator = DataCollatorForLanguageModeling(
          tokenizer=tokenizer, mlm=False,
  )
  return train_dataset, test_dataset, data_collator

train_dataset, test_dataset, data_collator = load_dataset(train_path, test_path, tokenizer)

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [54]:
training_args = TrainingArguments(
    output_dir="./gpt2-history-of-mysore",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_steps=400,
    save_steps=100,
    save_total_limit=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [55]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=24, training_loss=8.159687042236328, metrics={'train_runtime': 17.3254, 'train_samples_per_second': 40.98, 'train_steps_per_second': 1.385, 'total_flos': 23189667840000.0, 'train_loss': 8.159687042236328, 'epoch': 2.0})

In [56]:
trainer.save_model()

In [57]:
input_text = "Mysore was the  "
input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

In [58]:
output = model.generate(input_ids, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

In [59]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Mysore was thesor thesor thesor thesor thesor thesor thesor thesor thesor thesor thesor thesor thesor thes thesor thesor thesor thesor thesor thesor thes thesor thesor thes thesor thesor thesor thesor thesor thes thes thesor thes the


In [60]:
from huggingface_hub import notebook_login, create_repo, Repository
notebook_login()


In [61]:
repo_name = "fine-tuned-gpt2-history-of-mysore"  # Change this to your desired repository name
from huggingface_hub import HfApi

# Initialize the HfApi instance
api = HfApi(token="")

# Create a new repository
username = api.whoami()['name']  # Get your Hugging Face username
full_repo_name = f"{username}/{repo_name}"

# Create the repository (you can also create it on the Hugging Face website)
api.create_repo(repo_name, private=False)

api.upload_folder(
    folder_path='./gpt2-history-of-mysore',  # Path to the folder with your model
    repo_id=full_repo_name,  # Model repository name
    commit_message="GPT-2 Mysore"
)

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

scheduler.pt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

Upload 8 LFS files:   0%|          | 0/8 [00:00<?, ?it/s]

optimizer.pt:   0%|          | 0.00/996M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

events.out.tfevents.1724444676.cb2082bc24f6.2441.1:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/metriccoders/fine-tuned-gpt2-history-of-mysore/commit/b16b8deaebe154b4535f97a158e8838b68e8ec76', commit_message='GPT-2 Mysore', commit_description='', oid='b16b8deaebe154b4535f97a158e8838b68e8ec76', pr_url=None, pr_revision=None, pr_num=None)